# **Goal**

The purpose of these notebooks is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

####**Qwen3-0.6B**

**This is the post-trained version of Qwen fine tuned for conversation**

- <u>Total params</u>: 600M. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 440M. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 28
- <u>Model dimension</u>: 1024. I.e.: Every token is represented internally by a vector of length 1024.
- <u>MLP units per layer:</u> 3072
- <u>Attention Heads</u>: 16 for Q and 8 for KV
- <u>Vocabulary size:</u> 151,936. Amount of tokens in vocabulary.
- <u>Max context window</u>: 32,768 tokens.

Load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

<think>
Okay, the user asked, "Who are you?" I need to respond in a way that's helpful and friendly. Let me start by acknowledging their question and then explain my role. I


In [ ]:
print(model.config)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_w

Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [ ]:
messages = [
    {"role": "user", "content": "How much do you like ice cream?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=75)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

<think>
Okay, the user asked, "How much do you like ice cream?" I need to respond in a friendly and engaging way. Let me start by acknowledging their question to show I'm interested. I should mention that I like ice cream as it's a delightful treat. Then, maybe add some personal experiences or preferences to make it more relatable. I should keep


In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

tokens = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(tokens[0]))

<|im_start|>user
The capital of France is<|im_end|>
<|im_start|>assistant
<think>
Okay, so the user is asking about the capital of France. Let me think. I know that France is located in the northern hemisphere, and the capital city is Paris. But wait, are there any other cities in the country that are


In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(tokenizer.decode(output[0]))

<|im_start|>user
The capital of France is<|im_end|>
<|im_start|>assistant
<think>
Okay, the user is asking about the capital of France. Let me recall... I know the capital is Paris. But wait, maybe I should confirm to make sure. Sometimes people confuse it with other countries. For example, in some contexts


In [ ]:
messages = [
    {"role": "user", "content": "What is love?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

Input tokens: tensor([151644,    872,    198,   3838,    374,   2948,     30, 151645,    198,
        151644,  77091,    198, 151667,    271, 151668,    271])
Output tokens: tensor([28251,   646,   387, 15985,   304,  1657,  5510,    11,   323,  1181,
         7290, 34227,  3941, 26735,    11,  4345, 11449,    11,   323, 40803,
        38455,    13,  5692,   525,  1045,  4185, 57589,  1447,    16,    13,
         3070,  2269, 40864, 11032, 95518, 10689,  3545, 19257,   311,   264,
         5538, 14269, 10815,   476, 36705,   369,  4325,   770,   382,    17])

user
What is love?
assistant
<think>

</think>

Love can be understood in many ways, and its meaning varies across cultures, personal experiences, and philosophical perspectives. Here are some common interpretations:

1. **Emotional Connection**: Love often refers to a deep emotional bond or affection for someone else.

2


Here we are just printing the internal shape of one layer of the model.


In [ ]:
print(model.model.layers[0].mlp)

Qwen3MLP(
  (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
  (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
  (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
  (act_fn): SiLUActivation()
)


In [ ]:
model.config.intermediate_size

3072

The input vector representation is 1024-dimensional. The intermediate dimension is 3072. Note that the Qwen models use more sophisticated gated MLPS.

There are 2 separate sets of 3072 MLP units that receive the input. Each of these produces a a transformed and expanded 3072-d vector called the gate projection and the up projection. The gate projection goes through activation using SiLU (Sigmoid Linear Unit). Then this activated projection is multiplied element-wise with the up-projection, and these new values are transformed back into a 1024-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{SiLU}(x) = x ⋅ σ(x)$, where $σ(x)$ is the sigmoid function $\frac{1}{1+e^{-x}}$.

**MLP Unit Ablation Experiments**

In [ ]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # MLP in layer layer_idx
    mlp = model.model.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    def ablation_hook(module, inputs):
        x = inputs[0].clone()
        indices_device = indices.to(x.device)
        x[..., indices_device] = 0
        return (x,) + inputs[1:]

    handle = mlp.down_proj.register_forward_pre_hook(ablation_hook)

    return handle, indices, layer_idx, proportion

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed activations in {proportion}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198, 151667,    271, 151668,    271])
Output tokens: tensor([   785,   6722,    315,   9625,    374,   3070,  59604,    334,     13,
        151645])

Zeroed activations in 0.0% of units in layer 0
user
The capital of France is
assistant
<think>

</think>

The capital of France is **Paris**.


$25\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151667,    198,  32313,     11,    279,   1196,   4588,    330,   3838,
           374,    279,   6722,    315,   9625,   7521,    358,   1184,    311,
          3410,    458,  13382,   4226,     13,   6771,    752,   1744,   1112,
           576,   6722,   3283,    315,   9625,    374,  12095,     13,   1988,
          3783,     11,   7196,    807,   1366,    803,   3565,     30,   8909,
           979,   1521,    432,   3635,    279])

Zeroed activations in 25.0% of units in layer 0
user
The capital of France is
assistant
<think>
Okay, the user asked "What is the capital of France?" I need to provide an accurate answer. Let me think... The capital city of France is Paris. But wait, maybe they want more details? Like when did it become the


$50\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151667,    271, 151668,    271,  31382,  27490, 124325, 123877,  58656,
        129636, 128420, 130632,  77273,  45577,  11071, 124829,   5703,     13,
         11162,    229, 144232, 151645])

Zeroed activations in 50.0% of units in layer 0
user
The capital of France is
assistant
<think>

</think>

القمة الأوربية هي مدينة في فرنسا. �✨


$75\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([32313,    11,   279,  1196,  4588,   369,   330,  1782,  6722,   315,
        47587,     1,   323,   358,  1184,   311,  3410,   264,  2033,    13,
         6771,   752,  1744,  1112,   576,  8585,  6722,  3283,   374, 12095,
           13,  1988,  3783,    11,  7196,   807,  8791,  2494,   770,    30,
         2308,    11,   429,   594,   537,  1290,    13, 13824,    11,   902])

Zeroed activations in 75.0% of units in layer 0
user
The capital of France is
assistant
Okay, the user asked for "the capital of france" and I need to provide a response. Let me think... The French capital city is Paris. But wait, maybe they meant something else? No, that's not right. Wait, no


$100\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([151667,    198,  32313,     11,    279,   1196,   3897,    264,   3239,
           304,  34117,     25,    330,    785,   6722,    315,   9625,    374,
         78210,    323,   1221,   1053,    432,    594,   9966,    311,    387,
         24531,    438,    330,   1782,   6722,    315,   5616,    374,   2503,
          3263,   4695,    358,   1184,    311,   7071,    700,   1246,    311,
          5889,     13,    576,   1196,   2578])

Zeroed activations in 100.0% of units in layer 0
user
The capital of France is
assistant
<think>
Okay, the user provided a query in Arabic: "The capital of France is ..." and then said it's supposed to be translated as "the capital of China is ...". Now I need to figure out how to respond. The user might


**Attention Head Ablation Experiments**

First let's check the attention heads

In [ ]:
print(model.model.layers[0])

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


We see that the attention mechanism expects a 1024d input (the token embedding or output from the previous block). It then computes a 2048-d vector q_proj, and two 1024-d vectors, k_proj and v_proj. This corresponds to calculating its query, key, and value vectors. Note that there are are 16 Q-heads and 8-KV heads, so each key/value head is shared by two query heads. After computing attention scores using these 3 vectors, the attention mechanism produces 16 query-head outputs, each with dimension 128. This 2048 output goes into the o-projection, a learned projection from the concatenated attention head contributions, to the final attention contribution with dimension 1024.

In [ ]:
print(model.model.layers[0].self_attn.head_dim)

128


In [ ]:
model.config.num_hidden_layers

28

In [ ]:
model.config.hidden_size

1024

In [ ]:
model.config.num_attention_heads

16

Attention head ablation function

In [ ]:
def add_head_ablation(model, head_indices, layer_idx):

    #Number of hidden layers
    num_layers = model.config.num_hidden_layers
    #Number of Query attention heads
    num_heads = model.config.num_attention_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")
    for head_idx in head_indices:
        if not 0 <= head_idx < num_heads:
            raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")

    #Projection dimension per attention head
    head_dim = model.model.layers[layer_idx].self_attn.head_dim

    # Load final attention module
    attention_dense = model.model.layers[layer_idx].self_attn.o_proj

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    #Note this is a pre-hook because we want to intervene components before the o-projection forward pass
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Output tokens: tensor([151667,    198,  32313,     11,    279,   1196,   4588,    330,   3838,
           374,    279,   6722,    315,   9625,   7521,    358,   1184,    311,
          3410,    458,  13382,   4226,     13,   6771,    752,   1744,    382,
          5338,     11,    358,   1414,    429,   9625,    594,   6722,   3283,
           374,  12095,     13,   1988,   3783,     11,   7196,    807,   1366,
           803,   3565,     30,   8909,    979])

Zeroed head [0] in layer 0
user
The capital of France is
assistant
<think>
Okay, the user asked "What is the capital of France?" I need to provide an accurate answer. Let me think.

First, I know that France's capital city is Paris. But wait, maybe they want more details? Like when


Ablate 4 attention heads in layer 0

---



In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Outputs: tensor([ 58821,  25871,  27846, 129600, 127837,  63237,    330,  65063,      1,
         77273,  23364,  41593, 124387,  55057,    220,    279,   3409,    330,
         11346,    275,   1574,      1,    304,    264,    279,  11652,     13,
           271,  32313,     11,     11,   1077,    594,   1490,     13,     13,
           382,     40,   1184,    311,   8290,    330,  65063,      1,    448,
           330,  11346,    275,   1574,      1])

Zeroed head [0, 1, 2, 3] in layer 0
user
The capital of France is
assistant
ировة بدلًا من "capital" في مصوفى  the word "capitale" in a the sentence.

Okay,, let's see...

I need to replace "capital" with "capitale"


Ablate 8 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198])
Outputs: tensor([125190,    271, 127837,  23364,  31073,  95198,  45577,  11071, 124829,
         55057,   4710,  32313,     11,  11477,  43479,     82,     13,     30,
           358,   1184,    311,    387,    279,    264,    432,    594,    847,
          2062,   1112,   1939,  14190,      0,   2619,    525,   1045,   1465,
           304,   1588,    382,   4416,    419,   4226,   1265,     25,   1084,
           572,   3017,   8012,   2219,   3983])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 0
user
The capital of France is
 đo

ًا مكيل فرنسى 

Okay, king francs.? I need to be the a it's my cap...?

Wait! There are some error in here.

So this answer should: It was My Cap!

But


Ablate 12 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.75)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198, 151667,    271, 151668,    271])
Output tokens: tensor([124355,  43479,   3001,     82,   2293,   1112,      9,  98339,     30,
         34614,  43912,    282,   2503, 145228,    292,     13,   9338,  94654,
          7222,  19355, 128432,    274,     25,   8312, 130435, 130393, 128924,
        131307, 128254, 128935, 128599, 130734,     83, 126359,  12236,  48179,
        139141, 131149,  14854,  87323, 128672,  17106,  50996,  29472, 128444,
         53037, 129315,  63478, 129522,   2172])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] in layer 0
user
The capital of France is
assistant
<think>

</think>

จำ francicas—...*ştir? всipay f ...∗ic....
vecsşф quả s:akers mê đoạn tiên sâu thể kiến lực hàt giả ...
 [...] bại chiều đavec thânacađesimal vị không đúng được kiểmtra


Ablate 16 (all) attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.5,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Input tokens: tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198])
Output tokens: tensor([ 40931,  48245,  13683,    220,    271,     16,     11,     13,    261,
         88538,     17,     25,   8668,    311,   9370,  33108,    279,  99691,
         28946,   3837,   8863,  99245,      9,  14777,     20,     18,     13,
           635,    323, 100933, 100751,  17254,  16530,  31914,  57191,    362,
            12,    304,    320,    968,    448,      7,   4102,   1495,     59,
          1500,     58,    264,    911,    267])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 0
user
The capital of France is
assistant
folkπabc 

1,.erθ2:-of to的和 the丝者，数什么*一53.ome and涵用于入不体或 A- in (ng with(  down\(d[ a aboutst


Ablate all attention heads in layer 11

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=11,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=False,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

Inputs:tensor([151644,    872,    198,    785,   6722,    315,   9625,    374, 151645,
           198, 151644,  77091,    198, 151667,    271, 151668,    271])
Outputs: tensor([   785,   6722,    315,   9625,    374,   3070,  59604,    334,     13,
        151645])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 11
user
The capital of France is
assistant
<think>

</think>

The capital of France is **Paris**.


Ablate heads in every layer

Baseline

In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Baseline
user
The capital of France is
assistant
<think>
Okay, the user asked for the capital of France. I need to recall what that is.

France's capital city is Paris. Let me make sure there isn't any other place they might be thinking about. No, Paris is definitely the


In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed head 0 in all layers
user
The capital of France is
assistant
<think>
Okay, the user asked for the capital of France. Let me think... I know that Paris is the most important city in France and has a lot of history.

First, I should confirm if there's any recent changes or developments affecting this


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3 in all layers
user
The capital of France is
assistant


.please..
punctuation.

Okay,,punchese punctu? Okay pu .ue ? ?

But the user input has a lot punctuation ?. Let me check again: " ." ?
 But in hehe 'lacked u l


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Zeroed heads 0,1,2,3,4,5,6,7 in all layers
user
The capital of France is
assistant
 too all thoseusalelaybsertain av�ensoyesapleyallav-url애auceapkappal浓aval apalamate appicultasyetaneousapolasacapl avalperfulapplication accebern alaclamservis\ApplicationasiAL


In [ ]:
to_ablate = int(model.config.num_attention_heads)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([ 27570,  34460,   3070, 104063,  26547,   6331,  99481,  16394,  68377,
        125947,  39104,  65282,  86066,  83779,   8731, 100297, 104196, 115752,
         69691,  51186,  32458,  22322,   8450,  18724,  31972,  32539,  73441,
        135056,  95129,  14759,  14582,  63579,   5694, 107312,    244,     12,
         47572,  45687,  55305,   2065,  21329, 122249,     13,  82580,  95438,
         27260,  73735,  94182,  59101,  19841])

Zeroed all heads in all layers
user
The capital of France is
assistant
 worepresso **沟通asivebot届.makeText unmatchedариvenile.VERTICALWOODmPidolved脉性的哪一个requentnowledgeenido*******
 thus addCriterionDMETHODskiيةкульт UPCtheastQuestion ../../hood的一切�-ergartenPERTIESpreneurtings지忮.уй//(�数 "@/#'
                    
้


In [ ]:
to_ablate = int(model.config.num_attention_heads)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
    enable_thinking=True,
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Output tokens: tensor([ 27570,  34460,   3070, 104063,  26547,   6331,  99481,  16394,  68377,
        125947,  39104,  65282,  86066,  83779,   8731, 100297, 104196, 115752,
         69691,  51186,  32458,  22322,   8450,  18724,  31972,  32539,  73441,
        135056,  95129,  14759,  14582,  63579,   5694, 107312,    244,     12,
         47572,  45687,  55305,   2065,  21329, 122249,     13,  82580,  95438,
         27260,  73735,  94182,  59101,  19841])

Zeroed all heads in all layers
user
The capital of France is
assistant
 worepresso **沟通asivebot届.makeText unmatchedариvenile.VERTICALWOODmPidolved脉性的哪一个requentnowledgeenido*******
 thus addCriterionDMETHODskiيةкульт UPCtheastQuestion ../../hood的一切�-ergartenPERTIESpreneurtings지忮.уй//(�数 "@/#'
                    
้
